# T2.5 - DBRepo Data Load

Owner: Person C

Verify the T2.1 schema in 3NF, load raw data into DBRepo, verify T2.4 views.

## 1. Schema Verification

The T2.1 schema has three tables: station, time_dimension, weather_measurement.

**1NF:** Atomic columns, single primary keys (station_num, time_id, measurement_id). No repeating groups.

**2NF:** Station metadata (nuts_code, coordinates) moved to station table - depends only on station_num, not on time. Time metadata (year, month) moved to time_dimension - depends only on time_id, not on station. No partial dependencies on the composite key.

**3NF:** No transitive dependencies. In station, all columns depend directly on station_num. In time_dimension, all columns depend directly on time_id. In weather_measurement, measurements depend on the observation identified by measurement_id.

The raw CSV has 29 flat columns. Station codes (NUTS, DISTRICT_CODE, SUB_DISTRICT_CODE) repeat across every row for the same station. Year/month metadata repeats across every row for the same month. Under 3NF, these are separated into the dimension tables, with weather_measurement holding only observations and foreign keys. This eliminates redundancy while preserving all information.

**Conclusion:** The schema correctly models the data in 3NF.

In [ ]:
import pandas as pd
import numpy as np
from dbrepo.RestClient import RestClient

ENDPOINT = "https://test.dbrepo.tuwien.ac.at"
USERNAME = "azra1558"
PASSWORD = "Katalizator1558!"
DATABASE_ID = "899bfcba-7fec-40c9-9076-3a3a9372c844"

TABLE_IDS = {
    "weather_measurement": "2212bed4-ef8f-4d95-bb65-20b2adb28abd",
    "time_dimension": "9f4dc236-81cc-4bdb-93e7-a8b6ed5436f2",
    "station": "e6779029-ce40-4a9a-ad17-147e183dc757",
}

client = RestClient(endpoint=ENDPOINT, username=USERNAME, password=PASSWORD)
print(f"Connected to DBRepo. Database: {DATABASE_ID}")

## 2. Read Raw CSV

File has a description header on row 1, column names on row 2. Uses ; as delimiter and , as decimal separator.

In [ ]:
raw_df = pd.read_csv(
    "../data/raw/weather_raw_vienna_hohewarte_v1.csv.csv",
    sep=";", decimal=",", skiprows=1
)
print(f"Shape: {raw_df.shape}, Columns: {list(raw_df.columns)}")
print(f"Years: {raw_df['REF_YEAR'].min()}-{raw_df['REF_YEAR'].max()}")
print(f"Stations: {sorted(raw_df['STAT_NUM'].unique())}")
raw_df.head(2)

## 3. Transform into 3NF Tables

In [ ]:
# 3.1 Station table
station_df = (
    raw_df[["STAT_NUM", "NUTS", "DISTRICT_CODE", "SUB_DISTRICT_CODE"]]
    .drop_duplicates()
    .rename(columns={
        "STAT_NUM": "station_num", "NUTS": "nuts_code",
        "DISTRICT_CODE": "district_code", "SUB_DISTRICT_CODE": "sub_district_code"
    })
)
station_df["station_name"] = "Wien - Hohe Warte"
station_df["latitude_deg"] = 48.248611
station_df["longitude_deg"] = 16.356944
station_df["altitude_m"] = 202.0
station_df = station_df.drop_duplicates(subset=["station_num"]).set_index("station_num")
print(f"Station rows: {len(station_df)}")
station_df

In [ ]:
# 3.2 Time dimension table
raw_df["ref_month"] = pd.to_datetime(raw_df["REF_DATE"].astype(str), format="%Y%m").dt.month
raw_df["ref_year"] = raw_df["REF_YEAR"].astype(int)
raw_df["time_id"] = raw_df["ref_year"] * 100 + raw_df["ref_month"]

time_df = (
    raw_df[["time_id", "ref_year", "ref_month"]]
    .drop_duplicates().sort_values("time_id").set_index("time_id")
)
print(f"Time dimension rows: {len(time_df)}, range: {time_df.index.min()}-{time_df.index.max()}")
time_df.head(3)

In [ ]:
# 3.3 Weather measurement table
weather_df = raw_df.copy()
weather_df["station_num"] = weather_df["STAT_NUM"].astype(int)
weather_df["time_id"] = weather_df["ref_year"] * 100 + weather_df["ref_month"]

weather_df = weather_df.rename(columns={
    "T": "t_mean_c", "T_MAX": "t_max_c", "T_MIN": "t_min_c",
    "MEAN_T_MAX": "mean_t_max_c", "MEAN_T_MIN": "mean_t_min_c",
    "P": "p_mean_hpa", "P_MAX": "p_max_hpa", "P_MIN": "p_min_hpa",
    "PRECP_SUM": "precp_sum_mm", "NUM_PRECP_01": "num_precp_01",
    "REL_HUM": "rel_hum_pct", "REL_HUM_MAX": "rel_hum_max_pct", "REL_HUM_MIN": "rel_hum_min_pct",
    "WIND_VEL": "wind_vel_ms", "WIND_VEL_MAX": "wind_vel_max_ms", "NUM_WIND_VEL60": "num_wind_vel60",
    "SUN_H": "sun_h", "NUM_CLEAR": "num_clear", "NUM_CLOUD": "num_cloud",
    "NUM_FROST": "num_frost", "NUM_ICE": "num_ice", "NUM_SUMMER": "num_summer", "NUM_HEAT": "num_heat",
})

meas_cols = ["station_num", "time_id",
    "t_mean_c", "t_max_c", "t_min_c", "mean_t_max_c", "mean_t_min_c",
    "p_mean_hpa", "p_max_hpa", "p_min_hpa",
    "precp_sum_mm", "num_precp_01",
    "rel_hum_pct", "rel_hum_max_pct", "rel_hum_min_pct",
    "wind_vel_ms", "wind_vel_max_ms", "num_wind_vel60",
    "sun_h", "num_clear", "num_cloud",
    "num_frost", "num_ice", "num_summer", "num_heat"]

weather_df = weather_df[meas_cols].sort_values(["station_num", "time_id"]).reset_index(drop=True)
weather_df.insert(0, "measurement_id", range(1, len(weather_df) + 1))
weather_df = weather_df.set_index("measurement_id")
print(f"Weather rows: {len(weather_df)}")

## 4. Data Types and Missing Values

Structural NAs: REL_HUM_MAX/MIN, WIND_VEL_MAX, NUM_WIND_VEL60 missing before 1951; SUN_H missing before 1921.

Note: The DBRepo table `weather_measurement` was created (in T2.1) with `nullable=False` on all columns.
This means DBRepo rejects NULL values. We fill structural NAs with 0 before upload as a workaround.
The SQL schema in `sql/create_schema.sql` defines these columns as nullable (no `NOT NULL`),
so there is a mismatch between the SQL spec and the actual DBRepo table.

In [ ]:
int_cols = ["station_num", "time_id", "num_precp_01", "num_wind_vel60",
            "num_clear", "num_cloud", "num_frost", "num_ice", "num_summer", "num_heat"]
for col in int_cols:
    weather_df[col] = pd.to_numeric(weather_df[col], errors="coerce").astype("Int64")

dec_cols = ["t_mean_c", "t_max_c", "t_min_c", "mean_t_max_c", "mean_t_min_c",
            "p_mean_hpa", "p_max_hpa", "p_min_hpa", "precp_sum_mm",
            "rel_hum_pct", "rel_hum_max_pct", "rel_hum_min_pct",
            "wind_vel_ms", "wind_vel_max_ms", "sun_h"]
for col in dec_cols:
    weather_df[col] = pd.to_numeric(weather_df[col], errors="coerce")

# Fill structural NAs with 0 because DBRepo table has nullable=False on all columns
weather_df = weather_df.fillna(0)

nulls = weather_df.isnull().sum()
print("Null counts after fill:")
print(nulls[nulls > 0] if nulls.any() else "No remaining nulls")


## 5. Referential Consistency

In [ ]:
assert weather_df.index.is_unique and time_df.index.is_unique and station_df.index.is_unique
assert set(weather_df["time_id"]).issubset(set(time_df.index))
assert set(weather_df["station_num"]).issubset(set(station_df.index))

print(f"Referential checks passed.")
print(f"  station: {len(station_df)}, time: {len(time_df)}, weather: {len(weather_df)}")

## 6. Upload to DBRepo

The tables already exist (T2.1). Using import_table_data to insert rows.

In [ ]:
# Check column order from DBRepo
for name, tid in TABLE_IDS.items():
    tbl = client.get_table(DATABASE_ID, tid)
    print(f"{name}: {[c.name for c in tbl.columns]}")

In [ ]:
# Upload station
try:
    sdf = station_df.reset_index()[["station_num", "nuts_code", "district_code",
        "sub_district_code", "station_name", "latitude_deg", "longitude_deg", "altitude_m"]]
    client.import_table_data(DATABASE_ID, TABLE_IDS["station"], sdf)
    print("station uploaded")
except Exception as e:
    print(f"station upload (may already exist): {e}")


In [ ]:
# Upload time_dimension
try:
    tdf = time_df.reset_index()[["time_id", "ref_year", "ref_month"]]
    client.import_table_data(DATABASE_ID, TABLE_IDS["time_dimension"], tdf)
    print("time_dimension uploaded")
except Exception as e:
    print(f"time_dimension upload (may already exist): {e}")


In [ ]:
# Upload weather_measurement
try:
    wdf = weather_df.reset_index()
    wdf = wdf[["measurement_id", "station_num", "time_id",
        "t_mean_c", "t_max_c", "t_min_c", "mean_t_max_c", "mean_t_min_c",
        "p_mean_hpa", "p_max_hpa", "p_min_hpa",
        "precp_sum_mm", "num_precp_01",
        "rel_hum_pct", "rel_hum_max_pct", "rel_hum_min_pct",
        "wind_vel_ms", "wind_vel_max_ms", "num_wind_vel60",
        "sun_h", "num_clear", "num_cloud",
        "num_frost", "num_ice", "num_summer", "num_heat"]]
    client.import_table_data(DATABASE_ID, TABLE_IDS["weather_measurement"], wdf)
    print("weather_measurement uploaded")
except Exception as e:
    print(f"weather_measurement upload error: {e}")


## 7. Verify Row Counts

In [ ]:
print("Row counts after upload:")
for name, tid in TABLE_IDS.items():
    cnt = client.get_table_data_count(DATABASE_ID, tid)
    print(f"  {name}: {cnt}")

## 8. Verify T2.4 View

In [ ]:
views = client.get_views(DATABASE_ID)
for v in views:
    vc = client.get_view_data_count(DATABASE_ID, v.id)
    print(f"{v.name}: {vc} rows")

view = next(v for v in views if v.name == "weather_measurement_features")
view_count = client.get_view_data_count(DATABASE_ID, view.id)
table_count = client.get_table_data_count(DATABASE_ID, TABLE_IDS["weather_measurement"])
print(f"\nView == Table rows? {view_count == table_count} ({view_count} vs {table_count})")

## 9. Save Processed Files

In [ ]:
station_df.reset_index().to_csv("../data/processed/station_v1.csv", index=False)
time_df.reset_index().to_csv("../data/processed/time_dimension_v1.csv", index=False)
weather_df.reset_index().to_csv("../data/processed/weather_measurement_v1.csv", index=False)
print("Saved to data/processed/")